# 🛍️ Shopping Mall Customer Segmentation
## Member 3 — DBSCAN Clustering
**Dataset:** Shopping_Mall_Customer_Segmentation_Data_.csv  
**Algorithm:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise)  
**Task:** Unsupervised Machine Learning — Customer Segmentation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Load Pre-processed Data
> Data was already cleaned and scaled in `step0_eda_preprocessing.ipynb`.


In [ ]:
# Load preprocessed data from step0_eda_preprocessing.ipynb output
# ⚠️ Make sure you have run step0_eda_preprocessing.ipynb first!
import numpy as np
import pandas as pd

X_scaled = np.load('X_scaled.npy')
df_clean  = pd.read_csv('data_preprocessed.csv')
df        = pd.read_csv('Shopping_Mall_Customer_Segmentation_Data_.csv')

print('✅ Loaded X_scaled.npy   shape:', X_scaled.shape)
print('✅ Loaded data_preprocessed.csv shape:', df_clean.shape)
print('✅ Loaded original CSV   shape:', df.shape)

In [ ]:
# Use a sample for speed
sample_size = min(3000, len(X_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_idx]

MIN_SAMPLES = 5  # typical starting point = 2 * n_features
nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_sample)
distances, _ = nbrs.kneighbors(X_sample)
distances = np.sort(distances[:, -1])  # k-th nearest distance

plt.figure(figsize=(9, 5))
plt.plot(distances, color='steelblue', linewidth=1.5)
plt.xlabel('Data Points (sorted)')
plt.ylabel(f'{MIN_SAMPLES}-NN Distance')
plt.title('K-Distance Plot — Find the Elbow for Optimal eps')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('dbscan_kdistance.png', dpi=150)
plt.show()

print(f'Suggested eps range (based on elbow): look for inflection point above')

## 6. Hyperparameter Tuning — eps & min_samples Grid

In [ ]:
print('DBSCAN Parameter Search:')
print(f'{"eps":>6} | {"min_samples":>11} | {"n_clusters":>10} | {"noise %":>8} | {"silhouette":>11}')
print('-' * 60)

best_sil = -1
best_params = {}
results = []

for eps in [0.3, 0.5, 0.7, 1.0, 1.2, 1.5]:
    for min_s in [3, 5, 10]:
        db = DBSCAN(eps=eps, min_samples=min_s)
        labels = db.fit_predict(X_scaled)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct = (labels == -1).sum() / len(labels) * 100
        if n_clust >= 2:
            # Silhouette ignores noise points (label=-1)
            mask = labels != -1
            if mask.sum() > 1:
                sil = silhouette_score(X_scaled[mask], labels[mask])
            else:
                sil = -1
        else:
            sil = -1
        results.append({'eps': eps, 'min_samples': min_s, 'n_clusters': n_clust,
                        'noise_pct': noise_pct, 'silhouette': sil})
        print(f'{eps:>6.1f} | {min_s:>11} | {n_clust:>10} | {noise_pct:>7.1f}% | {sil:>11.4f}')
        if sil > best_sil and n_clust >= 2:
            best_sil = sil
            best_params = {'eps': eps, 'min_samples': min_s}

print(f'\n✅ Best params: eps={best_params["eps"]}, min_samples={best_params["min_samples"]} | Silhouette={best_sil:.4f}')

## 7. Train Final DBSCAN Model

In [ ]:
dbscan = DBSCAN(eps=best_params['eps'], min_samples=best_params['min_samples'])
cluster_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = (cluster_labels == -1).sum()

print(f'eps             : {best_params["eps"]}')
print(f'min_samples     : {best_params["min_samples"]}')
print(f'Clusters found  : {n_clusters}')
print(f'Noise points    : {n_noise} ({n_noise/len(df)*100:.1f}%)')
print()
print('Cluster Distribution:')
unique, counts = np.unique(cluster_labels, return_counts=True)
for c, cnt in zip(unique, counts):
    label = 'NOISE' if c == -1 else f'Cluster {c}'
    print(f'  {label}: {cnt} customers ({cnt/len(df)*100:.1f}%)')

df_clean['Cluster'] = cluster_labels
df['Cluster'] = cluster_labels

## 8. Evaluation Metrics

In [ ]:
mask = cluster_labels != -1
sil = silhouette_score(X_scaled[mask], cluster_labels[mask]) if mask.sum() > 1 else -1
dbi = davies_bouldin_score(X_scaled[mask], cluster_labels[mask]) if mask.sum() > 1 else -1

print('=' * 50)
print('       DBSCAN EVALUATION METRICS')
print('=' * 50)
print(f'  eps                 : {best_params["eps"]}')
print(f'  min_samples         : {best_params["min_samples"]}')
print(f'  Number of Clusters  : {n_clusters}')
print(f'  Noise Points        : {n_noise} ({n_noise/len(df)*100:.1f}%)')
print(f'  Silhouette Score    : {sil:.4f}  (closer to 1 = better)')
print(f'  Davies-Bouldin Index: {dbi:.4f}  (closer to 0 = better)')
print('  Note: Metrics computed on non-noise points only')
print('=' * 50)

## 9. Cluster Visualization (PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained Variance by 2 PCA components: {pca.explained_variance_ratio_.sum()*100:.2f}%')

unique_labels = sorted(set(cluster_labels))
palette = sns.color_palette('tab10', n_clusters)
color_map = {c: palette[i] for i, c in enumerate(l for l in unique_labels if l != -1)}

plt.figure(figsize=(9, 6))

# Plot noise points first (grey)
noise_mask = cluster_labels == -1
if noise_mask.sum() > 0:
    plt.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1],
                color='lightgray', label='Noise', alpha=0.3, s=10, marker='x')

# Plot clusters
for c in unique_labels:
    if c == -1:
        continue
    mask = cluster_labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                color=color_map[c], label=f'Cluster {c}', alpha=0.5, s=15)

plt.title(f'DBSCAN Clustering ({n_clusters} clusters + noise) — PCA 2D View')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.savefig('dbscan_pca_clusters.png', dpi=150)
plt.show()

## 10. Cluster Profile Analysis

In [ ]:
# Exclude noise for profile analysis
df_no_noise = df[df['Cluster'] != -1]
profile = df_no_noise.groupby('Cluster')[['Age', 'Annual Income', 'Spending Score']].mean().round(2)
profile['Count'] = df_no_noise.groupby('Cluster')['Age'].count()
profile['% of Total'] = (profile['Count'] / len(df) * 100).round(2)
print('Cluster Profiles (excluding noise):')
profile

In [ ]:
# Heatmap
plt.figure(figsize=(8, max(3, n_clusters * 0.5 + 2)))
heatmap_data = df_no_noise.groupby('Cluster')[['Age', 'Annual Income', 'Spending Score']].mean()
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='Blues', linewidths=0.5)
plt.title('DBSCAN Cluster Profile Heatmap')
plt.tight_layout()
plt.savefig('dbscan_heatmap.png', dpi=150)
plt.show()

## 11. Summary

In [ ]:
print('=' * 55)
print('          DBSCAN CLUSTERING — FINAL SUMMARY')
print('=' * 55)
print(f'  Algorithm             : DBSCAN')
print(f'  eps                   : {best_params["eps"]}')
print(f'  min_samples           : {best_params["min_samples"]}')
print(f'  Clusters Found        : {n_clusters}')
print(f'  Noise Points          : {n_noise} ({n_noise/len(df)*100:.1f}%)')
print(f'  Total Customers       : {len(df)}')
print(f'  Silhouette Score      : {sil:.4f}')
print(f'  Davies-Bouldin Index  : {dbi:.4f}')
print('=' * 55)
print('\nKey advantages of DBSCAN:')
print('  - Finds clusters of arbitrary shape')
print('  - Automatically labels outliers as noise')
print('  - Does NOT require specifying K in advance')